<a href="https://colab.research.google.com/github/aditya950820/Chotu/blob/main/notebooks/colab_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chotu on Colab: setup

Open this from GitHub each time (File → Open notebook → GitHub → `aditya950820/Chotu`), so you always run the latest code.

**One-time per Google account:** add these in Colab's 🔑 **Secrets** panel (left sidebar) and turn on *Notebook access*:
`HF_TOKEN`, `WANDB_API_KEY`, `GIT_NAME` (your GitHub username), `GIT_EMAIL`, and optionally `GITHUB_TOKEN` (fine-grained, *Contents: read/write* on Chotu only) to push progress notes.

**Runtime:** Runtime → Change runtime type → **L4** or **A100**. The free T4 can't do bf16, which the training script needs.

Files on the Colab machine are wiped when the session ends. Anything worth keeping goes to Google Drive (`/MyDrive/chotu`) or Hugging Face.

In [ ]:
# 1. GPU check
import subprocess, torch
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout)
assert torch.cuda.is_available(), 'No GPU: Runtime → Change runtime type → L4 or A100'
assert torch.cuda.is_bf16_supported(), 'This GPU has no bf16 (T4?). Switch to L4 or A100.'
print('GPU OK, bf16 OK')

In [ ]:
# 2. Keys from Colab Secrets into the environment (never printed, never written to git)
import os
from google.colab import userdata
for k in ['HF_TOKEN','WANDB_API_KEY','GIT_NAME','GIT_EMAIL','GITHUB_TOKEN']:
    try: os.environ[k] = userdata.get(k)
    except Exception: print(f'missing secret: {k}' + (' (optional)' if k=='GITHUB_TOKEN' else ''))
os.environ.setdefault('WANDB_ENTITY','adityaagrawal-student'); os.environ.setdefault('WANDB_PROJECT','chotu')

In [ ]:
# 3. Persistent storage on Google Drive (data, checkpoints, reports)
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/chotu'
os.makedirs(DRIVE, exist_ok=True); os.environ['CHOTU_STORAGE'] = DRIVE
print('storage:', DRIVE)

In [ ]:
# 4. Get the latest code from GitHub (the single source of truth)
import os, subprocess
tok = os.environ.get('GITHUB_TOKEN')
url = f'https://{tok}@github.com/aditya950820/Chotu.git' if tok else 'https://github.com/aditya950820/Chotu.git'
if not os.path.exists('/content/Chotu'):
    subprocess.run(['git','clone',url,'/content/Chotu'],check=True)
%cd /content/Chotu
!git pull --rebase -q
!git config user.name "$GIT_NAME" && git config user.email "$GIT_EMAIL"
!git log -1 --format='latest commit: %h %an | %s'

In [ ]:
# 5. Install pinned packages (keep Colab's own CUDA torch)
!grep -v '^torch==' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt

In [ ]:
# 6. Setup check: keys, git identity, model size test
!python scripts/check_setup.py

In [ ]:
# 7. Read where the project stands
print(open('docs/PROGRESS.md').read()[:3000])

## At the end of a Colab session
Add your entry to `docs/PROGRESS.md` (or ask Claude for the text), then run the cell below. It needs the `GITHUB_TOKEN` secret; otherwise commit from your laptop.

In [ ]:
# Push your progress notes (commits go out as you, never as Claude)
!git add docs/ && git commit -m "Progress update from Colab" && git pull --rebase -q && git push